In [5]:
import os
os.environ["HF_HOME"] = "/home/sdef0001/iq38_scratch/nmdid/cache/huggingface"
os.environ["TRANSFORMERS_CACHE"] = "/home/sdef0001/iq38_scratch/nmdid/cache/transformers"
os.environ["XDG_CACHE_HOME"] = "/home/sdef0001/iq38_scratch/nmdid/cache/xdg"

# (recommended) disable Xet storage so it stops writing to ~/.cache/huggingface/xet
os.environ["HF_HUB_DISABLE_XET"] = "1"

In [29]:
from transformers import AutoConfig, AutoTokenizer
from src.model.llm.qwen import VLMQwenForCausalLM
import torch

MERGED = "./models/Med3DVLM-Qwen-2.5-7B-vqa-17"
CACHE  = "/home/sdef0001/iq38_scratch/nmdid/cache/transformers"

cfg = AutoConfig.from_pretrained(MERGED, local_files_only=True)
print("cfg.vocab_size:", cfg.vocab_size, "num_new_tokens:", getattr(cfg, "num_new_tokens", None))

tok = AutoTokenizer.from_pretrained(MERGED, use_fast=False, cache_dir=CACHE, local_files_only=True)
print("len(tokenizer) =", len(tok))

model = VLMQwenForCausalLM.from_pretrained(MERGED, torch_dtype="auto", local_files_only=True)
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# sanity
assert all("lora_" not in n for n,_ in model.named_parameters())
assert model.get_input_embeddings().weight.size(0) == len(tok)

proj = [n for n,_ in model.named_parameters() if "mm_projector" in n]
print("mm_projector params:", len(proj))
assert len(proj) > 0

# ✅ pass 'inputs', not 'input_ids'
enc = tok("Hello", return_tensors="pt")
ids = enc["input_ids"].to(device)
mask = enc.get("attention_mask", None)
if mask is not None:
    mask = mask.to(device)

out = model.generate(
    inputs=ids,
    attention_mask=mask,   # harmless if your generate ignores it
    max_new_tokens=5
)
print(tok.decode(out[0], skip_special_tokens=True))

cfg.vocab_size: 151666 num_new_tokens: None
len(tokenizer) = 151666


Loading checkpoint shards: 100%|██████████| 4/4 [00:00<00:00, 21.53it/s]


mm_projector params: 48


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


, everyone, and,


In [30]:
import torch
from src.model.llm.qwen import VLMQwenForCausalLM

torch.set_grad_enabled(False)

MERGED  = "./models/Med3DVLM-Qwen-2.5-7B-vqa-17"
NONLORA = "./output/Med3DVLM-Qwen-2.5-7B-finetune_vqa_17/non_lora_weights.bin"

# 1) Load merged model on CPU (no GPU memory drama)
merged = VLMQwenForCausalLM.from_pretrained(MERGED, torch_dtype="auto", device_map=None)
merged.to("cpu")
msd = merged.state_dict()  # use state_dict so buffers/params keys match saving conventions

# 2) Load the saved non-LoRA weights and normalize keys same as your merge script
nl_sd = torch.load(NONLORA, map_location="cpu")

def norm_key(k: str) -> str:
    k = k.replace("base_model.", "")
    k = k.replace("model.model.", "model.")
    return k

nl_sd = {norm_key(k): v for k, v in nl_sd.items()}

# 3) Pick a few projector keys that exist in BOTH dicts
proj_keys = [k for k in nl_sd.keys() if "mm_projector" in k and k in msd]
proj_keys = sorted(proj_keys)[:3]
print("Checking:", proj_keys)
assert len(proj_keys) > 0, "No overlapping mm_projector keys found to compare!"

# 4) Compare values (cast to fp32 so bf16/fp16 don’t cause false mismatches)
for k in proj_keys:
    a = msd[k].detach().to(dtype=torch.float32, device="cpu")
    b = nl_sd[k].detach().to(dtype=torch.float32, device="cpu")
    if a.shape != b.shape:
        raise RuntimeError(f"Shape mismatch for {k}: merged {tuple(a.shape)} vs nl {tuple(b.shape)}")

    diff = (a - b).norm().item()
    print(f"{k:60s} shape={tuple(a.shape)}  |Δ|| = {diff:.6e}")
    # allow tiny numeric noise from dtype conversions / merges
    assert torch.allclose(a, b, atol=1e-5, rtol=1e-5), f"{k} mismatch beyond tolerance"

print("✅ non-LoRA projector weights match")

# 5) Sanity: ensure no LoRA tensors survived in the merged model
lora_leftovers = [k for k in msd.keys() if "lora_" in k]
assert not lora_leftovers, f"Merged model still has LoRA keys: {lora_leftovers[:5]}"

Loading checkpoint shards: 100%|██████████| 4/4 [00:00<00:00, 21.14it/s]


Checking: ['model.mm_projector.high_mixer.0.ln1.bias', 'model.mm_projector.high_mixer.0.ln1.weight', 'model.mm_projector.high_mixer.0.ln2.bias']
model.mm_projector.high_mixer.0.ln1.bias                     shape=(768,)  |Δ|| = 0.000000e+00
model.mm_projector.high_mixer.0.ln1.weight                   shape=(768,)  |Δ|| = 0.000000e+00
model.mm_projector.high_mixer.0.ln2.bias                     shape=(768,)  |Δ|| = 0.000000e+00
✅ non-LoRA projector weights match


In [34]:
import torch
from transformers import AutoTokenizer
from safetensors.torch import load_file
from peft import PeftModel, PeftConfig
from src.model.llm.qwen import VLMQwenForCausalLM

BASE   = "./models/Med3DVLM-Qwen-2.5-7B"
ADIR   = "./output/Med3DVLM-Qwen-2.5-7B-finetune_vqa_17/ADAPTER_ONLY"
MERGED  = "./models/Med3DVLM-Qwen-2.5-7B-vqa-17"
NONLORA = "./output/Med3DVLM-Qwen-2.5-7B-finetune_vqa_17/non_lora_weights.bin"

# tokenizer (same in both)
tok  = AutoTokenizer.from_pretrained(BASE, use_fast=False)
tok.add_special_tokens({"additional_special_tokens": ["<im_patch>"]})
if tok.pad_token is None and tok.unk_token is not None:
    tok.pad_token = tok.unk_token

# A) merged model
m1 = VLMQwenForCausalLM.from_pretrained(MERGED, torch_dtype="auto").eval()

# B) compose base + non_lora + adapter (no merge)
m2 = VLMQwenForCausalLM.from_pretrained(BASE, torch_dtype="auto")
# init vision/projector EXACTLY like your training/merge
# (mirror your merge_clean.py: set mm_projector_type/margs/etc. then call)
# For brevity here, assume your helper did it already in your codebase:
# m2.get_model().initialize_vision_modules(...)
# m2.initialize_vision_tokenizer(...)

# load non-lora
nl = torch.load(NONLORA, map_location="cpu")
def norm(k): return k.replace("base_model.","").replace("model.model.","model.")
nl = {norm(k):v for k,v in nl.items()}
m2.load_state_dict(nl, strict=False)

# attach adapter (no merge)
pc = PeftConfig.from_pretrained(ADIR)
m2 = PeftModel(m2, pc)
adapter_sd = load_file(os.path.join(ADIR, "adapter_model.safetensors"))
m2.load_state_dict(adapter_sd, strict=True)
m2.eval()

# Compare logits on same input
ids = tok("Sanity check", return_tensors="pt").input_ids
with torch.no_grad():
    logits1 = m1(input_ids=ids).logits
    logits2 = m2(input_ids=ids).logits

diff = (logits1 - logits2).float().abs().max().item()
print("max|Δ| logits:", diff)
assert diff < 1e-4, "Merged model not equivalent to base+adapter+nonlora!"
print("Behavioral equivalence ✅")


Loading checkpoint shards: 100%|██████████| 7/7 [03:33<00:00, 30.50s/it]


RuntimeError: Error(s) in loading state_dict for PeftModel:
	Missing key(s) in state_dict: "base_model.model.model.embed_tokens.weight", "base_model.model.model.layers.0.self_attn.q_proj.base_layer.weight", "base_model.model.model.layers.0.self_attn.q_proj.base_layer.bias", "base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight", "base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight", "base_model.model.model.layers.0.self_attn.k_proj.base_layer.weight", "base_model.model.model.layers.0.self_attn.k_proj.base_layer.bias", "base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight", "base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight", "base_model.model.model.layers.0.self_attn.v_proj.base_layer.weight", "base_model.model.model.layers.0.self_attn.v_proj.base_layer.bias", "base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight", "base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight", "base

In [ ]:
import torch
prompt = "Given the image: <im_patch><im_patch> What is shown?"
ids = tok(prompt, return_tensors="pt").input_ids

def try_shape(shape):
    try:
        img = torch.zeros(shape)  # fp32 ok for smoke test
        with torch.no_grad():
            out1 = m1.generate(inputs=ids, images=img, max_new_tokens=5)
            out2 = m2.generate(inputs=ids, images=img, max_new_tokens=5)
        print(shape, "OK")
        return True
    except Exception as e:
        print(shape, "failed:", e)
        return False

ok = try_shape((1,1,128,256,256)) or try_shape((1,1,256,256,128))
assert ok, "Vision smoke test failed for both shapes"


In [33]:
import os, torch
from src.model.llm.qwen import VLMQwenForCausalLM

torch.set_grad_enabled(False)

MERGED = "./models/Med3DVLM-Qwen-2.5-7B-vqa-17"
INIT_MM = "./output/Med3DVLM-Qwen-2.5-7B/mm_projector.bin"   # your pretrain projector path

def norm_key(k: str) -> str:
    return k.replace("base_model.", "").replace("model.model.", "model.")

# 1) merged (post-train) projector
merged = VLMQwenForCausalLM.from_pretrained(MERGED, torch_dtype="auto", device_map=None).to("cpu")
msd = merged.state_dict()
mproj = {k: v.float().cpu() for k, v in msd.items() if "mm_projector" in k}

# 2) initial (pre-train) projector
if not os.path.exists(INIT_MM):
    raise FileNotFoundError(f"Can't find initial projector at {INIT_MM}")
init_sd = torch.load(INIT_MM, map_location="cpu")
init_sd = {norm_key(k): v.float().cpu() for k,v in init_sd.items()}

common = sorted(set(mproj.keys()) & set(init_sd.keys()))
print("common projector tensors:", len(common))

changed, total_l2 = 0, 0.0
for k in common:
    a, b = mproj[k], init_sd[k]
    if a.shape != b.shape: 
        print("shape mismatch:", k, a.shape, b.shape); 
        continue
    diff = (a - b).norm().item()
    total_l2 += diff
    if diff > 1e-6:
        changed += 1
        # print(k, diff)  # uncomment to inspect per-tensor deltas

print(f"Changed tensors: {changed}/{len(common)}")
print(f"Total L2 delta across projector: {total_l2:.6e}")

Loading checkpoint shards: 100%|██████████| 4/4 [00:00<00:00, 21.29it/s]


common projector tensors: 48
Changed tensors: 40/48
Total L2 delta across projector: 6.578856e+00
